In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import numpy as np
import datetime
import time
import pickle
import os
import seaborn as sns
from scipy.interpolate import make_interp_spline
from tabulate import tabulate

import sys
sys.path.append("src")

from data_generation import *
from neural_networks import *
from estimation import *

pd.options.mode.chained_assignment = None
os.chdir('Result_Tables/')

## Appendix G - Table A5

### RCL

In [ ]:
seed_list = [7456,145609, 214341, 237639, 148649, 763490, 677253, 560404, 591830,
    84947,  26672, 184535, 874369,  39876, 738836, 100589, 812811,
    75122, 182935, 911677]

dg_list = [rcl_fe] 
JKM_list = [[5,0,100]]
delta = 0.1 

hyper_params = {'hidden_size': 256, 'num_hidden_layers': 5, 'n_epochs': 2000, 'learning_rate': 0.0001}

os.chdir("/mnt/sdb1/colab-project/rslt_fe")


for [J, K, M] in JKM_list:
    for dg in dg_list:
        for i in range(len(seed_list[1:3])):
            seed = seed_list[i]
            ## for FE, the total number of params becomes K+J+2 (K non-price features, 1 price, 1 intercept, J FE) 
            params = [np.random.normal(0,1,K+J+2) / (1 * 2), np.ones(K+J+2)]
            ## the parameter for price
            params[0][-1] = -1
            ## change the iteration to iteration_fe
            full_one_iteration_fe(J, M, K, seed, dg, params, hyper_params, delta)
            print(J, K, M, i, 'finished', datetime.datetime.now())
        report(J, K, M, seed_list, dg)



### MNL

In [ ]:
seed_list = [7456,145609, 214341, 237639, 148649, 763490, 677253, 560404, 591830,
        84947,  26672, 184535, 874369,  39876, 738836, 100589, 812811,
        75122, 182935, 911677]
    
dg_list = [mnl_choice_fe] 
JKM_list = [[5,0,100]]
delta = 0.1 

hyper_params = {'hidden_size': 256, 'num_hidden_layers': 5, 'n_epochs': 2000, 'learning_rate': 0.0001}

os.chdir("/mnt/sdb1/colab-project/rslt_fe")


for [J, K, M] in JKM_list:
    for dg in dg_list:
        for i in range(len(seed_list[1:3])):
            seed = seed_list[i]
            ## for FE, the total number of params becomes K+J+2 (K non-price features, 1 price, 1 intercept, J FE) 
            params =  np.ones(J+K+2)
            params[-1] = -1
            ## change the iteration to iteration_fe
            full_one_iteration_fe(J, M, K, seed, dg, params, hyper_params, delta)
            print(J, K, M, i, 'finished', datetime.datetime.now())
        report(J, K, M, seed_list, dg)


### Variations of RCL

In [ ]:
seed_list = [7456,145609, 214341, 237639, 148649, 763490, 677253, 560404, 591830,
        84947,  26672, 184535, 874369,  39876, 738836, 100589, 812811,
        75122, 182935, 911677]
    
dg_list = [rcl_in3_fe]#rcl_sin_fe,rcl_log_fe
JKM_list = [[5,0,100]]
delta = 0.1 

hyper_params = {'hidden_size': 256, 'num_hidden_layers': 5, 'n_epochs': 2000, 'learning_rate': 0.0001}

for [J, K, M] in JKM_list:
    for dg in dg_list:
        for i in range(len(seed_list[1:3])):
            seed = seed_list[i]
            ## for FE, the total number of params becomes K+J+2 (K non-price features, 1 price, 1 intercept, J FE) 
            params = [np.random.normal(0,1,K+J+2) / (1 * 2), np.ones(K+J+2)]
            ## the parameter for price
            params[0][-1] = -1
            ## change the iteration to iteration_fe
            full_one_iteration_fe(J, M, K, seed, dg, params, hyper_params, delta)
            print(J, K, M, i, 'finished', datetime.datetime.now())
        report(J, K, M, seed_list, dg)

## Combine results

In [8]:
os.chdir("/Result_Tables")

JKM_dg_list = [[5,0,100,rcl_fe],[5,0,100,mnl_choice_fe],[5,0,100,rcl_log_fe],[5,0,100,rcl_sin_fe], [5,0,100,rcl_in3_fe]]

seed_list = [7456,145609, 214341, 237639, 148649, 763490, 677253, 560404, 591830,
    84947,  26672, 184535, 874369,  39876, 738836, 100589, 812811,
    75122, 182935, 911677]

elas_median_df, elas_mae_df, elas_rmse_df, error_df  = output(JKM_dg_list)


In [9]:
error_rslt = error_df.groupby(['dg','J','M','K'])[['mae_deep','mae_mnl','mae_rcl','mae_single','mae_random',
                                                  'mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']].mean().reset_index()
error_rslt[['rmse_deep','rmse_mnl','rmse_rcl','rmse_single','rmse_random']] = np.sqrt(error_rslt[['mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']])
error_rslt['obs'] = error_rslt['M'] * error_rslt['J'] *  0.2
error_rslt = error_rslt[[ 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'mae_random','rmse_random',
                             'obs']]
error_rslt

,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,mae_random,rmse_random,obs
0,mnl_choice_fe,5,100,0,0.004758,0.061416,0.003976,0.089833,0.004356,0.066679,0.004746,0.061455,0.102219,0.285864,100.0
1,rcl_fe,5,100,0,0.002750,0.044227,0.024906,0.143263,0.003416,0.050353,0.009080,0.074274,0.075561,0.240569,100.0
2,rcl_in3_fe,5,100,0,0.015885,0.094643,0.023312,0.134802,0.021818,0.129169,0.027286,0.139534,0.083534,0.254051,100.0
3,rcl_log_fe,5,100,0,0.014808,0.076241,0.049847,0.238506,0.038580,0.167976,0.067287,0.164967,0.154025,0.313439,100.0
4,rcl_sin_fe,5,100,0,0.004163,0.053873,0.047134,0.193275,0.034460,0.150869,0.014177,0.098076,0.092107,0.267431,100.0


In [10]:
elas_rmse_df.columns = [col.replace('ae_', 'rmse_') if 'ae_' in col else col for col in elas_rmse_df.columns]
elas_mae_df.columns = [col.replace('ae_', 'mae_') if 'ae_' in col else col for col in elas_mae_df.columns]

elas_rslt = pd.merge(elas_mae_df, elas_rmse_df,
                     on = ['type','dg','J','M','K'])
elas_rslt['obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * 0.8).astype('int')
elas_rslt.loc[elas_rslt.type == 'cross','obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * (elas_rslt['J']-1) * 0.8).astype('int')

elas_rslt = elas_rslt[['type', 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'obs']]
elas_rslt

,type,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,obs
0,cross,rcl_fe,5,100,0,0.024572,0.064928,0.065193,0.186879,0.006496,0.021508,0.050078,0.113635,32000
1,self,rcl_fe,5,100,0,0.048839,0.140783,0.450009,1.284137,0.028852,0.073670,0.101570,0.288361,8000
2,cross,mnl_choice_fe,5,100,0,0.015060,0.032789,0.000925,0.001680,0.000290,0.000571,0.011806,0.045389,32000
3,self,mnl_choice_fe,5,100,0,0.023882,0.056041,0.004512,0.006244,0.001201,0.002337,0.011832,0.046433,8000
4,cross,rcl_log_fe,5,100,0,0.037753,0.112400,0.078917,0.205283,0.069345,0.564547,0.113680,0.371889,32000
5,self,rcl_log_fe,5,100,0,0.095202,0.297083,5.089238,5.116583,0.963823,1.606134,0.154521,0.465765,8000
6,cross,rcl_sin_fe,5,100,0,0.029077,0.095444,0.069026,0.263333,0.064020,0.247165,0.066006,0.183615,32000
7,self,rcl_sin_fe,5,100,0,0.055801,0.268885,0.533013,1.061212,0.246735,0.629944,0.145921,0.365520,8000
8,cross,rcl_in3_fe,5,100,0,0.084246,4.843119,0.070025,6.022960,0.043420,6.032073,0.188625,5.739117,32000
9,self,rcl_in3_fe,5,100,0,0.169415,4.692638,0.719329,2.014464,0.404429,2.027488,0.404674,2.014910,8000


## Appendix H - Table A6 RCL with bimodal mixture

In [ ]:
seed_list = [7456,145609, 214341, 237639, 148649, 763490, 677253, 560404, 591830,
        84947,  26672, 184535, 874369,  39876, 738836, 100589, 812811,
        75122, 182935, 911677]
    
dg_list = [rcl_mix] 
JKM_list = [[10,0,200]]
delta = 0.1 

hyper_params = {'hidden_size': 256, 'num_hidden_layers': 5, 'n_epochs': 2000, 'learning_rate': 0.0001}


for [J, K, M] in JKM_list:
    for dg in dg_list:
        for i in range(len(seed_list[1:3])):
            seed = seed_list[i]
            alpha = np.random.normal(0,1,1) / (1 * 2)

            b1 = [alpha, -0.1]
            b2 = [alpha, -2]
            sigma1 = [1, 0.1]
            sigma2 = [1, 2]
            pi = 0.5 

            params = [ b1, b2, sigma1, sigma2, pi ] 

            full_one_iteration(J, M, K, seed, dg, params, hyper_params, delta)
            data_file_name = 'data_'+ 'J' + str(J) + 'K' +  str(K) +'M' + str(M) + '_'+str(seed) + str(dg).split()[1] +".pickle"
            os.remove(data_file_name)
            print(J, K, M, i, 'finished', datetime.datetime.now())
        report(J, K, M, seed_list, dg)


In [11]:
JKM_dg_list = [[10,0,200,rcl_mix]]

seed_list = [7456,145609, 214341, 237639, 148649, 763490, 677253, 560404, 591830,
    84947,  26672, 184535, 874369,  39876, 738836, 100589, 812811,
    75122, 182935, 911677]

elas_median_df, elas_mae_df, elas_rmse_df, error_df  = output(JKM_dg_list)


In [12]:
error_rslt = error_df.groupby(['dg','J','M','K'])[['mae_deep','mae_mnl','mae_rcl','mae_single','mae_random',
                                                  'mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']].mean().reset_index()
error_rslt[['rmse_deep','rmse_mnl','rmse_rcl','rmse_single','rmse_random']] = np.sqrt(error_rslt[['mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']])
error_rslt['obs'] = error_rslt['M'] * error_rslt['J'] *  0.2

error_rslt = error_rslt[[ 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'mae_random','rmse_random',
                             'obs']]

error_rslt

,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,mae_random,rmse_random,obs
0,rcl_mix,10,200,0,0.000971,0.029685,0.020396,0.177891,0.005568,0.066302,0.018505,0.114092,0.030817,0.145461,400.0


In [13]:
elas_rmse_df.columns = [col.replace('ae_', 'rmse_') if 'ae_' in col else col for col in elas_rmse_df.columns]
elas_mae_df.columns = [col.replace('ae_', 'mae_') if 'ae_' in col else col for col in elas_mae_df.columns]

elas_rslt = pd.merge(elas_mae_df, elas_rmse_df,
                     on = ['type','dg','J','M','K'])
elas_rslt['obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * 0.8).astype('int')
elas_rslt.loc[elas_rslt.type == 'cross','obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * (elas_rslt['J']-1) * 0.8).astype('int')

elas_rslt = elas_rslt[['type', 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'obs']]
elas_rslt

,type,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,obs
0,cross,rcl_mix,10,200,0,0.018371,0.039576,0.061832,0.113243,0.030907,0.051056,0.159493,0.233217,288000
1,self,rcl_mix,10,200,0,0.062425,0.101396,0.605758,0.824358,0.231029,0.298496,0.393807,0.570175,32000


## Appendix I - Table A7

In [ ]:
seed_list = [7456,145609, 214341, 237639, 148649, 763490, 677253, 560404, 591830,
        84947,  26672, 184535, 874369,  39876, 738836, 100589, 812811,
        75122, 182935, 911677]
    
dg_list = [rcl_mix] 
JKM_list = [[10,0,200]]
delta = 0.1 

hyper_params = {'hidden_size': 256, 'num_hidden_layers': 5, 'n_epochs': 2000, 'learning_rate': 0.0001}


for [J, K, M] in JKM_list:
    for dg in dg_list:
        for i in range(len(seed_list[1:3])):
            seed = seed_list[i]
            alpha = np.random.normal(0,1,1) / (1 * 2)

            b1 = [alpha, -0.1]
            b2 = [alpha, -2]
            sigma1 = [1, 0.1]
            sigma2 = [1, 2]
            pi = 0.5 

            params = [ b1, b2, sigma1, sigma2, pi ] 

            full_one_iteration_keep_price(J, M, K, seed, dg, params, hyper_params, delta)
            data_file_name = 'data_'+ 'J' + str(J) + 'K' +  str(K) +'M' + str(M) + '_'+str(seed) + str(dg).split()[1] +".pickle"
            os.remove(data_file_name)
            print(J, K, M, i, 'finished', datetime.datetime.now())
        report(J, K, M, seed_list, dg)


In [15]:
os.chdir("/mnt/sdb1/colab-project/rslt_keep_price")

JKM_dg_list = [[10,0,200,rcl_mix]]

seed_list = [7456,145609, 214341, 237639, 148649, 763490, 677253, 560404, 591830,
    84947,  26672, 184535, 874369,  39876, 738836, 100589, 812811,
    75122, 182935, 911677]

for [J, K, M, dg] in JKM_dg_list:    
    report(J, K, M, seed_list, dg)

elas_median_df, elas_mae_df, elas_rmse_df, error_df  = output(JKM_dg_list)


In [16]:
error_rslt = error_df.groupby(['dg','J','M','K'])[['mae_deep','mae_mnl','mae_rcl','mae_single','mae_random',
                                                  'mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']].mean().reset_index()
error_rslt[['rmse_deep','rmse_mnl','rmse_rcl','rmse_single','rmse_random']] = np.sqrt(error_rslt[['mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']])
error_rslt['obs'] = error_rslt['M'] * error_rslt['J'] *  0.2

error_rslt = error_rslt[[ 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'mae_random','rmse_random',
                             'obs']]

error_rslt

,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,mae_random,rmse_random,obs
0,rcl_mix,10,200,0,0.000793,0.026759,0.020283,0.177757,0.005524,0.066246,0.001081,0.028084,0.030977,0.146456,400.0


In [18]:
elas_rmse_df.columns = [col.replace('ae_', 'rmse_') if 'ae_' in col else col for col in elas_rmse_df.columns]
elas_mae_df.columns = [col.replace('ae_', 'mae_') if 'ae_' in col else col for col in elas_mae_df.columns]

elas_rslt = pd.merge(elas_mae_df, elas_rmse_df,
                     on = ['type','dg','J','M','K'])
elas_rslt['obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * 0.8).astype('int')
elas_rslt.loc[elas_rslt.type == 'cross','obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * (elas_rslt['J']-1) * 0.8).astype('int')

elas_rslt = elas_rslt[['type', 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'obs']]
elas_rslt

,type,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,obs
0,cross,rcl_mix,10,200,0,0.021698,0.043512,0.061858,0.113561,0.030625,0.050563,0.249990,0.366037,288000
1,self,rcl_mix,10,200,0,0.065896,0.107023,0.607452,0.826826,0.228339,0.294231,0.629542,0.872493,32000


## Appendix J - Table A8

In [32]:
seed_list = [7456,145609, 214341, 237639, 148649, 763490, 677253, 560404, 591830,
        84947,  26672, 184535, 874369,  39876, 738836, 100589, 812811,
        75122, 182935, 911677]
    
dg_list = [rcl] 
JKM_list = [[5,10,100]]
delta = 0.1 

hyper_params = {'hidden_size': 256, 'num_hidden_layers': 5, 'n_epochs': 2000, 'learning_rate': 0.0001}

for [J, K, M] in JKM_list:
    for dg in dg_list:
        for i in range(len(seed_list)):
            seed = seed_list[i]
            params = [np.random.normal(0,1,K+2) / (K * 2), np.ones(K+2)]
            params[0][-1] = -1
            full_one_iteration(J, M, K, seed, dg, params, hyper_params, delta)
            print(J, K, M, i, 'finished', datetime.datetime.now())
        report(J, K, M, seed_list, dg)


Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
5 10 100 0 finished 2026-03-05 20:17:12.730708
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
5 10 100 1 finished 2026-03-05 20:24:55.219509
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
5 10 100 2 finished 2026-03-05 20:32:20.727503
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculated.
5 10 100 3 finished 2026-03-05 20:39:09.319132
Data is generated.
Main model is trained.
RCL is estimated.
MNL is estimated.
NP is estimated.
Prediction errors are calculated.
Elasticity errors are calculate

In [27]:
os.getcwd()

'/mnt/sdb1/colab-project/singleNN'

In [29]:
os.chdir("/home/jupyter-colab-project/replication/Full_replications/Result_Tables/Single_NN")

JKM_dg_list = [[5,10,100,rcl]]

seed_list = [7456,145609, 214341, 237639, 148649, 763490, 677253, 560404, 591830,
    84947,  26672, 184535, 874369,  39876, 738836, 100589, 812811,
    75122, 182935, 911677]

#for [J, K, M, dg] in JKM_dg_list:    
#    report(J, K, M, seed_list, dg)

elas_median_df, elas_mae_df, elas_rmse_df, error_df  = output(JKM_dg_list)


In [30]:
error_rslt = error_df.groupby(['dg','J','M','K'])[['mae_deep','mae_mnl','mae_rcl','mae_single','mae_random',
                                                  'mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']].mean().reset_index()
error_rslt[['rmse_deep','rmse_mnl','rmse_rcl','rmse_single','rmse_random']] = np.sqrt(error_rslt[['mse_deep','mse_mnl','mse_rcl','mse_single','mse_random']])
error_rslt['obs'] = error_rslt['M'] * error_rslt['J'] *  0.2
error_rslt = error_rslt[[ 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'mae_random','rmse_random',
                             'obs']]
error_rslt

,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,mae_random,rmse_random,obs
0,rcl,5,100,10,0.028924,0.151105,0.038127,0.175303,0.0042,0.057667,0.032599,0.160188,0.065254,0.231838,100.0


In [31]:
elas_rmse_df.columns = [col.replace('ae_', 'rmse_') if 'ae_' in col else col for col in elas_rmse_df.columns]
elas_mae_df.columns = [col.replace('ae_', 'mae_') if 'ae_' in col else col for col in elas_mae_df.columns]

elas_rslt = pd.merge(elas_mae_df, elas_rmse_df,
                     on = ['type','dg','J','M','K'])
elas_rslt['obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * 0.8).astype('int')
elas_rslt.loc[elas_rslt.type == 'cross','obs'] = (elas_rslt['M'] * elas_rslt['J'] * 20 * (elas_rslt['J']-1) * 0.8).astype('int')

elas_rslt = elas_rslt[['type', 'dg', 'J', 'M', 'K', 
                             'mae_deep', 'rmse_deep', 
                             'mae_mnl', 'rmse_mnl', 
                             'mae_rcl', 'rmse_rcl',  
                             'mae_single', 'rmse_single', 
                             'obs']]
elas_rslt

,type,dg,J,M,K,mae_deep,rmse_deep,mae_mnl,rmse_mnl,mae_rcl,rmse_rcl,mae_single,rmse_single,obs
0,cross,rcl,5,100,10,0.033660,0.074156,0.049242,0.074578,0.003001,0.007295,0.043535,0.095822,32000
1,self,rcl,5,100,10,0.107625,0.258670,0.147416,0.433553,0.012502,0.031101,0.115485,0.295150,8000
